# Détection de places de parking libres / occupées — YOLOv8 sur PKLot

Ce notebook entraîne un détecteur **YOLOv8n** qui repère chaque place de parking sur une image de caméra de surveillance et indique si elle est **libre** ou **occupée**.

**Étapes :** préparation du dataset PKLot → entraînement → évaluation sur le jeu de test → images de démonstration.

Tout ce qui doit être gardé (modèle `best.pt`, courbes, métriques, images de démo) est copié dans Google Drive, dans le dossier `resultats_yolov8`.

> Exécution : Colab avec GPU (*Exécution → Modifier le type d'exécution → GPU T4*). L'entraînement dure environ 1 h 15.

## 1. Installation et connexion à Google Drive

In [ ]:
!pip install -q ultralytics

import os, glob, json, random, shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import yaml
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

# Dossier du projet sur Google Drive (contient pklot-parking.zip).
# Si le dossier a été partagé avec vous, ajoutez-le à "Mon Drive" (clic droit → Organiser → Ajouter un raccourci)
# ou adaptez ce chemin.
DRIVE_DIR = Path('/content/drive/MyDrive/Proje IA et deep learning')
RESULTS_DIR = DRIVE_DIR / 'resultats_yolov8'
DATA_DIR = Path('/content/datasets/pklot')

assert (DRIVE_DIR / 'pklot-parking.zip').exists(), f"pklot-parking.zip introuvable dans {DRIVE_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Préparation du dataset

PKLot (Université Fédérale du Paraná, licence CC BY 4.0) contient 12 416 images de parkings prises par temps ensoleillé, nuageux et pluvieux. La version utilisée est l'export Roboflow au format YOLO (images 640×640).

On garde les **noms de classes d'origine** du fichier `data.yaml` et on corrige seulement les chemins.

In [ ]:
if not DATA_DIR.exists():
    shutil.unpack_archive(str(DRIVE_DIR / 'pklot-parking.zip'), str(DATA_DIR))

yaml_path = DATA_DIR / 'data.yaml'
config = yaml.safe_load(open(yaml_path))
print('Classes du dataset :', config['names'])

config.update({'path': str(DATA_DIR), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images'})
yaml.safe_dump(config, open(yaml_path, 'w'))

for split in ['train', 'valid', 'test']:
    print(f"{split:5} : {len(os.listdir(DATA_DIR / split / 'images'))} images")

## 3. Entraînement de YOLOv8n

In [ ]:
model = YOLO('yolov8n.pt')   # modèle pré-entraîné sur COCO (transfer learning)

results = model.train(
    data=str(yaml_path),
    epochs=25,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='pklot',
    exist_ok=True,
    plots=True,
)
run_dir = Path(results.save_dir)

# Sauvegarde sur Drive de ce qui doit survivre à la fin de la session Colab
for name in ['weights/best.pt', 'results.png', 'results.csv', 'confusion_matrix_normalized.png', 'BoxPR_curve.png']:
    src = run_dir / name
    if src.exists():
        shutil.copy(src, RESULTS_DIR / Path(name).name)
print('Fichiers copiés dans', RESULTS_DIR)

## 4. Évaluation sur le jeu de test (images jamais vues pendant l'entraînement)

In [ ]:
best = YOLO(str(RESULTS_DIR / 'best.pt'))
metrics = best.val(data=str(yaml_path), split='test', plots=False)

summary = {
    'precision': round(float(metrics.box.mp), 3),
    'recall': round(float(metrics.box.mr), 3),
    'mAP50': round(float(metrics.box.map50), 3),
    'mAP50-95': round(float(metrics.box.map), 3),
    'inference_ms_par_image': round(metrics.speed['inference'], 1),
}
json.dump(summary, open(RESULTS_DIR / 'metrics_test.json', 'w'), indent=2)
summary

## 5. Démonstration : compter les places libres et occupées

Places libres en **vert**, places occupées en **rouge**, compteur en haut de l'image.

In [ ]:
FREE, OCCUPIED = (60, 200, 60), (40, 40, 230)
roles = {i: ('occupied' if 'occup' in n.lower() else 'free') for i, n in best.names.items()}

def annotate(img_path):
    res = best.predict(str(img_path), conf=0.25, verbose=False)[0]
    frame = res.orig_img.copy()
    free = occ = 0
    for box in res.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        is_free = roles[int(box.cls)] == 'free'
        cv2.rectangle(frame, (x1, y1), (x2, y2), FREE if is_free else OCCUPIED, 2)
        free += is_free
        occ += not is_free
    label = f'Libres : {free}   Occupees : {occ}'
    cv2.rectangle(frame, (0, 0), (330, 34), (30, 30, 30), -1)
    cv2.putText(frame, label, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return frame

demo_dir = RESULTS_DIR / 'demo'
demo_dir.mkdir(exist_ok=True)
random.seed(1)
samples = random.sample(sorted((DATA_DIR / 'test' / 'images').glob('*.jpg')), 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, samples):
    frame = annotate(img_path)
    cv2.imwrite(str(demo_dir / img_path.name), frame)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'demo_grille.jpg', dpi=80, bbox_inches='tight')
plt.show()
print('Images de démo enregistrées dans', demo_dir)

## 6. Vérification concrète : le compteur de places est-il juste ?

Les métriques YOLO (précision, mAP) comparent des boîtes. Pour un utilisateur, la vraie question est plutôt : *le nombre de places libres affiché est-il juste ?* On compare donc, image par image du jeu de test, le nombre de places libres et occupées prédit avec le nombre réel donné par les annotations.

In [ ]:
roles = {i: ('occupied' if 'occup' in n.lower() else 'free') for i, n in best.names.items()}
free_id = [i for i in roles if roles[i] == 'free'][0]
occ_id = [i for i in roles if roles[i] == 'occupied'][0]

errors_free, errors_occ, exact = [], [], 0
test_images = sorted((DATA_DIR / 'test' / 'images').glob('*.jpg'))
for img_path in test_images:
    label_path = DATA_DIR / 'test' / 'labels' / (img_path.stem + '.txt')
    true_ids = [int(line.split()[0]) for line in open(label_path) if line.strip()] if label_path.exists() else []
    pred_ids = [int(c) for c in best.predict(str(img_path), conf=0.25, verbose=False)[0].boxes.cls]
    err_free = abs(pred_ids.count(free_id) - true_ids.count(free_id))
    err_occ = abs(pred_ids.count(occ_id) - true_ids.count(occ_id))
    errors_free.append(err_free)
    errors_occ.append(err_occ)
    exact += (err_free == 0 and err_occ == 0)

comptage = {
    'images_test': len(test_images),
    'erreur_moyenne_places_libres': round(sum(errors_free) / len(test_images), 2),
    'erreur_moyenne_places_occupees': round(sum(errors_occ) / len(test_images), 2),
    'pourcentage_images_compte_exact': round(100 * exact / len(test_images), 1),
    'erreur_max_places_libres': max(errors_free),
}
json.dump(comptage, open(RESULTS_DIR / 'metrics_comptage.json', 'w'), indent=2)
comptage